# First milestone: generate one student trajectory

On-policy distillation starts with a response sampled from the student's current distribution. In this milestone, we generate that response and preserve its token IDs for later work.

Run these cells from top to bottom in Google Colab. Select **Runtime → Change runtime type → GPU** first. We use one prompt, so every token tensor has a batch dimension of size **1**. No teacher or training is needed for this milestone.

## 1. Install the notebook dependencies

We use `uv` to install all six dependencies into the active notebook's Python environment. If `uv` is missing, we use pip once to install uv itself; uv then handles the project packages. Passing `sys.executable` to `--python` ensures packages go into the environment running this notebook, as described in the [uv environment documentation](https://docs.astral.sh/uv/pip/environments/).

Transformers 4.51 or newer supports Qwen3; we stay within version 4 for this walkthrough. If Colab asks you to restart the session after installation, restart it before continuing.

We install all six requested packages. `bitsandbytes`, `sentencepiece`, and `pandas` are available for later sections; this milestone does not use quantization or tables. See the [Qwen3 base model requirements](https://huggingface.co/Qwen/Qwen3-0.6B-Base#requirements).


In [2]:
import importlib.util
import subprocess
import sys

# sys.executable is the Python interpreter running this notebook's kernel.
# Bootstrap uv only if it is missing; pip is used solely to install uv itself.
if importlib.util.find_spec("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "uv"],
        check=True,
    )

# uv installs into this kernel's environment, avoiding an unrelated local .venv.
# torch provides GPU tensors; transformers provides tokenization and generation.
# accelerate supports GPU placement; the remaining packages are for later milestones.
# No model is loaded here: these dependencies prepare us to sample a student trajectory.
# check=True stops this cell if installation fails, so the failure is visible.
subprocess.run(
    [
        sys.executable, "-m", "uv", "pip", "install",
        "--python", sys.executable,
        "torch", "transformers>=4.51.0,<5", "accelerate",
        "bitsandbytes", "sentencepiece", "pandas",
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'uv', 'pip', 'install', '--python', '/usr/bin/python3', 'torch', 'transformers>=4.51.0,<5', 'accelerate', 'bitsandbytes', 'sentencepiece', 'pandas'], returncode=0)

## 2. Check the GPU and choose a numeric precision

This notebook requires a CUDA GPU. We print its name and total memory, then select BF16 when supported and FP16 otherwise. These are 16-bit formats for the model's floating-point weights; token IDs remain integers.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Stop early with an actionable message if Colab is still using a CPU runtime.
assert torch.cuda.is_available(), (
    "CUDA is required. In Colab, select Runtime > Change runtime type > GPU."
)

# device identifies the single GPU used for both the student and its prompt tensors.
device = torch.device("cuda:0")
gpu_properties = torch.cuda.get_device_properties(device)
gpu_memory_gib = gpu_properties.total_memory / (1024 ** 3)
bf16_supported = torch.cuda.is_bf16_supported()

# BF16 has a wider numeric range than FP16; use it when the GPU supports it.
# This dtype controls the student's weights, not the integer token IDs.
model_dtype = torch.bfloat16 if bf16_supported else torch.float16

print(f"GPU name: {gpu_properties.name}")
print(f"GPU memory (total): {gpu_memory_gib:.2f} GiB")
print(f"BF16 supported: {bf16_supported}")
print(f"Selected model dtype: {model_dtype}")

GPU name: NVIDIA A100-SXM4-40GB
GPU memory (total): 39.49 GiB
BF16 supported: True
Selected model dtype: torch.bfloat16


## 3. Load the tokenizer and the student

The tokenizer converts text into token IDs and back. We load it from `Qwen/Qwen3-1.7B`, but the only model weights loaded are those of the student, `Qwen/Qwen3-0.6B-Base`.

Evaluation mode disables training-specific behavior such as dropout. It still allows random token sampling during generation. The student will produce the trajectory used in later distillation steps.

In [2]:
# These identifiers distinguish the tokenizer files from the student model weights.
tokenizer_model_id = "Qwen/Qwen3-1.7B"
student_model_id = "Qwen/Qwen3-0.6B-Base"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_id)

# A pad token marks unused positions in a padded batch. This one-prompt batch
# needs no padding, but generate() still receives an explicit pad-token ID.
# Reuse the end-of-sequence (EOS) token only if the tokenizer lacks a pad token.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load all student weights onto GPU 0 in the selected 16-bit floating-point format.
# No teacher weights, adapters, or quantization are involved in this milestone.
student_model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    torch_dtype=model_dtype,
    device_map={"": device.index},
)
student_model.eval()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

## 4. Tokenize and inspect the prompt

The base student receives the exact question as plain text, without a chat template. Tokens may represent words, pieces of words, punctuation, or spaces.

`prompt_inputs` contains integer `input_ids` and an `attention_mask`, both shaped `[1, prompt_length]`. The mask tells the model which positions contain real input. Every token here belongs to the **prompt**; no response exists yet.

In [3]:
# prompt_text is the question that conditions this student's sampled trajectory.
prompt_text = (
    "A store has 24 boxes. Each box contains 15 pencils. The store sells 87 pencils. "
    "How many pencils remain? Explain your reasoning step by step."
)

# Return PyTorch tensors and move them to the same GPU as the student.
# add_special_tokens=False preserves the plain prompt without extra wrapper tokens.
# With one unpadded prompt, the attention_mask has shape [1, prompt_length]
# and contains only ones: all positions belong to the actual question.
prompt_inputs = tokenizer(
    prompt_text,
    return_tensors="pt",
    add_special_tokens=False,
).to(device)

# The tokenizer returns input_ids with shape:
# [batch_size, number_of_prompt_tokens], which here is [1, prompt_length].
# Keep these original prompt IDs so we can check the generated sequence's prefix.
prompt_token_ids = prompt_inputs["input_ids"]
prompt_length = prompt_token_ids.shape[1]

print("Prompt:", prompt_text)
print("Prompt token IDs:", prompt_token_ids[0].tolist())
print("input_ids tensor shape:", tuple(prompt_token_ids.shape))
print("Number of prompt tokens:", prompt_length)

Prompt: A store has 24 boxes. Each box contains 15 pencils. The store sells 87 pencils. How many pencils remain? Explain your reasoning step by step.
Prompt token IDs: [32, 3553, 702, 220, 17, 19, 14697, 13, 8886, 3745, 5610, 220, 16, 20, 96338, 13, 576, 3553, 30778, 220, 23, 22, 96338, 13, 2585, 1657, 96338, 7146, 30, 81917, 697, 32711, 3019, 553, 3019, 13]
input_ids tensor shape: (1, 36)
Number of prompt tokens: 36


## 5. Sample one response from the student

A **trajectory** here is the ordered sequence of response tokens sampled from the student, conditioned on the prompt. Generating it with the current student makes it on-policy.

We allow up to 128 new tokens; an end-of-sequence token can stop generation earlier. The fixed seed helps reproduce this sample with the same hardware and software setup. Evaluation mode and sampling serve different purposes: the former disables training behavior, while the latter chooses tokens randomly from the student's distribution.

We disable top-k and top-p filtering so they do not truncate the distribution. See the [Hugging Face generation settings](https://huggingface.co/docs/transformers/main_classes/text_generation). The returned tensor contains both the prompt and the response.

In [4]:
# Fixing the random seed lets us reproduce the same sampled trajectory
# when rerunning the notebook with the same hardware and software setup.
torch.manual_seed(42)

# This milestone only generates tokens. inference_mode() avoids building an
# automatic-differentiation graph, reducing memory use during generation.
with torch.inference_mode():
    complete_sequence_ids = student_model.generate(
        **prompt_inputs,
        # Limit the RESPONSE to 128 new tokens; the prompt is not part of this budget.
        max_new_tokens=128,
        # Sampling is required because we want a trajectory drawn from the
        # student's current probability distribution. This is what makes the
        # resulting response on-policy.
        do_sample=True,
        # Temperature 1.0 leaves the model's probability distribution unchanged.
        temperature=1.0,
        # Disable candidate filtering so sampling uses the full distribution.
        top_k=0,
        top_p=1.0,
        # One prompt produces one sampled sequence, without beam search.
        num_beams=1,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id,
        # Return only token IDs; no per-token scoring tensors are requested.
        return_dict_in_generate=False,
        output_scores=False,
    )

# complete_sequence_ids has shape [1, prompt_length + response_length].
# Its first prompt_length positions are the original question; the rest
# are tokens sampled by the student, which we will preserve in the next cell.

## 6. Separate and preserve the response tokens

Split at `prompt_length` along the sequence dimension. The prompt is the context; only the newly generated tokens form the response trajectory. Keep the batch dimension so `response_token_ids` has shape `[1, response_length]`.

The checks below confirm that generation preserved the prompt and produced a nonempty response of the expected length. Decoding is for human inspection only: preserve the original token IDs for later scoring. An EOS token, if generated, stays in those IDs and counts toward the response length even though it is hidden in the decoded text.

In [5]:
# generate() returns the original prompt followed by newly generated tokens.
# Therefore, everything beginning at prompt_length belongs to the response.
# The colon keeps the batch dimension: response_token_ids is [1, response_length].
response_token_ids = complete_sequence_ids[:, prompt_length:]
response_length = response_token_ids.shape[1]
total_sequence_length = complete_sequence_ids.shape[1]

# These checks protect the prompt/response boundary needed for later distillation:
# the prompt is conditioning context, and the response is the student's trajectory.
assert response_length > 0, "The student must generate at least one response token."
assert torch.equal(
    complete_sequence_ids[:, :prompt_length], prompt_token_ids
), "The complete sequence must begin with the original prompt token IDs."
assert response_length == total_sequence_length - prompt_length, (
    "The response-token count must equal total length minus prompt length."
)

# [0] selects the single response for decoding, with shape [response_length].
# Skipping special tokens makes text easier to read without modifying the IDs.
student_response_text = tokenizer.decode(
    response_token_ids[0], skip_special_tokens=True
)
print("Decoded student response:\n" + student_response_text)
print("Generated response token IDs:", response_token_ids[0].tolist())
print("Response-token tensor shape:", tuple(response_token_ids.shape))
print("Number of response tokens:", response_length)
print("Total sequence length (prompt + response):", total_sequence_length)

# Keep response_token_ids and complete_sequence_ids in notebook memory for the
# next milestone. Re-tokenizing decoded text could change the original trajectory.

Decoded student response:
 To determine the number of pencils remaining, follow these steps:

1. **Calculate the total number of pencils in the store:**
   
   The store has 24 boxes, and each box contains 15 pencils.
   
   \[
   \text{Total pencils} = 24 \text{ boxes} \times 15 \text{ pencils/box} = 360 \text{ pencils}
   \]

2. **Subtract the number of pencils sold:**
   
   The store sold 87 pencils.
   
   \[
   \text{Pencils remaining} = 3
Generated response token IDs: [2014, 8253, 279, 1372, 315, 96338, 9664, 11, 1795, 1493, 7354, 1447, 16, 13, 3070, 47866, 279, 2790, 1372, 315, 96338, 304, 279, 3553, 25, 1019, 5872, 256, 576, 3553, 702, 220, 17, 19, 14697, 11, 323, 1817, 3745, 5610, 220, 16, 20, 96338, 624, 5872, 256, 1124, 9640, 256, 1124, 1318, 90, 7595, 96338, 92, 284, 220, 17, 19, 1124, 1318, 90, 14697, 92, 1124, 15136, 220, 16, 20, 1124, 1318, 90, 96338, 14, 2011, 92, 284, 220, 18, 21, 15, 1124, 1318, 90, 96338, 532, 256, 1124, 2533, 17, 13, 3070, 3136, 2144, 279, 1372, 31

## What we accomplished

After running the cells above:

- The **student** produced the trajectory by sampling its own response to the prompt.
- The trajectory consists of the generated response token IDs in `response_token_ids`; the prompt is its conditioning context.
- The teacher has not been used yet. Loading its tokenizer does not load a teacher model.
- No model parameters have been updated.
- The next step will make the student calculate a log probability for every token it generated.

The original prompt, complete sequence, and response token IDs remain available in notebook memory. This milestone stops at generating and preserving one student trajectory.

# Part 2: Student token log probabilities

Run Part 1 first so its model and sampled token tensors are available. We now ask the **same student** to score the exact trajectory it generated. We reuse the token IDs directly rather than decode and re-tokenize the response.

Earlier cells named the model `student_model`; here `student` is another name for that same object. No model is loaded again. The student remains in evaluation mode, and this part does not update its weights.


## 1. Pass the complete sequence through the student

`complete_sequence_ids` contains the prompt followed by the response, with shape `[batch_size, sequence_length]`. Our batch size is 1, and `sequence_length = prompt_length + response_length`.

A forward pass returns **logits**: unnormalized scores for possible next tokens. Their shape is `[batch_size, sequence_length, vocabulary_size]`. At every sequence position, there is one score for each token in the vocabulary. The model's causal attention prevents a position from seeing future tokens, even though we provide the whole sequence at once.

`torch.no_grad()` avoids recording gradients. We do not pass labels, request a loss, or perform an update.


In [6]:
# Reuse the already loaded model under the requested name; this does not copy weights.
student = student_model

# This single sequence has no padding: every position is a real prompt or response
# token. The attention mask has shape [batch_size, sequence_length] and is all ones.
# The model additionally applies its own causal mask to hide future tokens.
complete_attention_mask = torch.ones_like(complete_sequence_ids)

# Re-score the student's existing trajectory without recording gradients.
# use_cache=False avoids storing a generation cache for this one forward pass.
with torch.no_grad():
    student_forward_result = student(
        input_ids=complete_sequence_ids,
        attention_mask=complete_attention_mask,
        use_cache=False,
        return_dict=True,
    )

# logits[b, t, v] is the score for vocabulary token v as the NEXT token after
# sequence position t in batch item b. These scores are not yet probabilities.
logits = student_forward_result.logits
batch_size, sequence_length, vocabulary_size = logits.shape
print("Logits shape [batch_size, sequence_length, vocabulary_size]:", tuple(logits.shape))
print("Dimension 0 — batch_size:", batch_size)
print("Dimension 1 — sequence_length (prompt + response):", sequence_length)
print("Dimension 2 — vocabulary_size:", vocabulary_size)


Logits shape [batch_size, sequence_length, vocabulary_size]: (1, 164, 151936)
Dimension 0 — batch_size: 1
Dimension 1 — sequence_length (prompt + response): 164
Dimension 2 — vocabulary_size: 151936


## 2. Align each prediction with its next-token target

Logits at sequence position **t** predict the token at position **t + 1**, not the token at position t. We therefore pair `logits[:, :-1, :]` with `complete_sequence_ids[:, 1:]`.

- Remove the final logits position because its next token lies beyond the stored sequence.
- Remove the first token from the targets because there is no preceding logits position in this forward pass to predict it.

The resulting `shifted_logits` has shape `[batch_size, sequence_length - 1, vocabulary_size]`, and `target_token_ids` has shape `[batch_size, sequence_length - 1]`. Shifted position 0 now pairs the logits at original position 0 with the actual token at original position 1.


In [ ]:
# Keep predictions only where the next token exists in our stored trajectory.
# Shape: [batch_size, sequence_length - 1, vocabulary_size].
shifted_logits = logits[:, :-1, :]

# Each target is one token ahead of the logits position that predicts it.
# Shape: [batch_size, sequence_length - 1]; IDs remain integer-valued.
# These targets still include prompt tokens (except the first) and all response tokens.
target_token_ids = complete_sequence_ids[:, 1:]

print("Shifted logits shape:", tuple(shifted_logits.shape))
print("Target token IDs shape:", tuple(target_token_ids.shape))


## 3. Select the log probability of each token that appeared

`log_softmax` converts the vocabulary scores into natural-log probabilities. We first use `.float()` to perform this calculation in FP32, improving numerical stability when the model uses BF16 or FP16. This does not change the dtype of the model's weights.

For each position, `all_token_log_probs` contains a score for **every** vocabulary token. `target_token_ids` tells us which token actually appeared. `gather` selects only that token's score. See the PyTorch references for [log-softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html) and [gather](https://docs.pytorch.org/docs/stable/generated/torch.gather.html).

These are conditional probabilities: each token is scored given the prompt and all preceding response tokens. We are scoring the existing trajectory, not sampling a new one.


In [ ]:
# Convert BF16/FP16 logits to FP32 before normalization for better numerical stability.
# dim=-1 normalizes over vocabulary choices, not over positions in the sequence.
# Shape remains [batch_size, sequence_length - 1, vocabulary_size].
all_token_log_probs = torch.log_softmax(
    shifted_logits.float(),
    dim=-1,
)

# all_token_log_probs stores a log probability for every vocabulary token.
# target_token_ids identifies the token that actually appeared at each next position.
# unsqueeze(-1) makes indices [batch_size, sequence_length - 1, 1], so gather
# retrieves one vocabulary entry per position. squeeze(-1) removes only that last
# size-1 dimension, leaving [batch_size, sequence_length - 1].
selected_token_log_probs = all_token_log_probs.gather(
    dim=-1,
    index=target_token_ids.unsqueeze(-1),
).squeeze(-1)

print("Selected token log-probability shape:", tuple(selected_token_log_probs.shape))


## 4. Keep only response-token scores

This is the important one-token offset. The first response token is at index `prompt_length` in `complete_sequence_ids`, but the logits that predict it are at index **`prompt_length - 1`**: the final prompt position.

After the shift, target index `j` refers to original token index `j + 1`. Therefore, the first response token is at shifted target index `prompt_length - 1`. Starting at `prompt_length` instead would accidentally discard the first response token's score.

| Original logits position | Original target token position | Token being scored |
| --- | --- | --- |
| `prompt_length - 1` | `prompt_length` | First response token |
| `prompt_length` | `prompt_length + 1` | Second response token, if present |

Slicing from `prompt_length - 1` excludes prompt-token **targets**, while retaining the final prompt position's prediction of the first response token. We should obtain `[batch_size, response_length]` scores. Exponentiating these log probabilities is only for the human-readable display.


In [ ]:
# Original response tokens begin at prompt_length, but shifted targets begin
# one position earlier. The final prompt position predicts the first response token.
response_start_in_shifted_sequence = prompt_length - 1

# Exclude scores whose targets are prompt tokens; keep every response target,
# including an EOS token if one was generated. Shape: [batch_size, response_length].
response_student_log_probs = selected_token_log_probs[
    :,
    response_start_in_shifted_sequence:
]

# Undo the natural logarithm for display only. These are individual conditional
# token probabilities, not a distribution over response positions or a training loss.
# The shape is still [batch_size, response_length].
response_student_probs = response_student_log_probs.exp()

print("Response log-probability shape:", tuple(response_student_log_probs.shape))
print("Expected response-token count:", response_length)


## 5. Check token alignment and numerical values

Matching lengths is necessary, but it is not enough: we also compare the shifted target IDs directly with `response_token_ids`. This verifies that each retained score belongs to the correct generated token.

Log probabilities must be finite and no greater than zero; the displayed probabilities must lie between zero and one. A success message appears only after every assertion passes.


In [ ]:
# The shift must remove exactly one sequence position, and each remaining target
# must have exactly one selected score. All dimensions include the batch axis.
assert shifted_logits.shape[1] == complete_sequence_ids.shape[1] - 1
assert target_token_ids.shape == selected_token_log_probs.shape

# Check both the number and identity of response targets. This catches an
# off-by-one slice even if a table of decoded tokens happens to look plausible.
assert response_student_log_probs.shape[1] == response_length
assert torch.equal(
    target_token_ids[:, response_start_in_shifted_sequence:],
    response_token_ids,
)

# Valid log probabilities are finite and nonpositive. Exponentiating them gives
# probabilities in [0, 1]; these checks also catch NaNs and invalid numeric results.
assert torch.isfinite(response_student_log_probs).all()
assert (response_student_log_probs <= 0).all()
assert (response_student_probs >= 0).all()
assert (response_student_probs <= 1).all()

print("All student token-alignment tests passed.")


## 6. Inspect the response token by token

The table has one row per generated response token. `response_position` starts at zero within the response, not within the complete sequence. We decode each token individually and use `repr()` so spaces and newline escapes remain visible. Individual tokens can be word fragments; their displays are for inspection, while the original IDs remain authoritative.

The full table is preserved in `response_token_table`; we display only its first 30 rows. The mean, minimum, and maximum summarize **all** response-token log probabilities. They describe this sampled trajectory, not a training objective or a measure of whether the answer is correct.


In [ ]:
import pandas as pd
from IPython.display import display

# This notebook generated one response. Select batch item 0 and move the small
# per-token tensors from GPU to CPU lists so pandas can display them.
# Each list contains response_length entries; no prompt tokens are included.
response_token_id_list = response_token_ids[0].cpu().tolist()
response_log_probability_list = response_student_log_probs[0].cpu().tolist()
response_probability_list = response_student_probs[0].cpu().tolist()

# Decode one ID at a time. repr() makes whitespace visible, including '\n'.
# Keep special tokens visible too: an emitted EOS still has its own score.
response_token_table = pd.DataFrame({
    "response_position": list(range(response_length)),
    "token_id": response_token_id_list,
    "decoded_token": [repr(tokenizer.decode([token_id])) for token_id in response_token_id_list],
    "student_log_probability": response_log_probability_list,
    "student_probability": response_probability_list,
})
display(response_token_table.head(30))

# These scalar summaries use every response token, not just the first 30 rows.
# item() converts a single-value PyTorch tensor into a Python number for printing.
print("Number of response tokens:", response_length)
print("Mean response-token log probability:", response_student_log_probs.mean().item())
print("Minimum response-token log probability:", response_student_log_probs.min().item())
print("Maximum response-token log probability:", response_student_log_probs.max().item())


## 7. Manually inspect the prompt-to-response boundary

Read the display below as an arrow: **the final prompt position's logits → the first response token**. All positions use zero-based indexing.

The model predicts that token using the entire prompt prefix, not just the final prompt token. We print the two boundary tokens, the predicting logits position, and the selected student log probability to make the one-token relationship explicit.


In [ ]:
# Batch item 0 is our only trajectory. The boundary lies between these two
# original sequence positions: prompt_length - 1 and prompt_length.
final_prompt_position = prompt_length - 1
first_response_position = prompt_length
final_prompt_token_id = complete_sequence_ids[0, final_prompt_position].item()
first_response_token_id = response_token_ids[0, 0].item()

# Response score [0, 0] came from logits[0, prompt_length - 1, :], selecting
# the vocabulary entry for first_response_token_id after log-softmax.
first_response_student_log_probability = response_student_log_probs[0, 0].item()

print(
    f"Final prompt token: position {final_prompt_position}, ID {final_prompt_token_id}, "
    f"text {repr(tokenizer.decode([final_prompt_token_id]))}"
)
print(
    f"  logits at position {final_prompt_position}  -->  "
    f"first response token at position {first_response_position}"
)
print(
    f"First response token: ID {first_response_token_id}, "
    f"text {repr(tokenizer.decode([first_response_token_id]))}"
)
print("Student log probability of that first response token:", first_response_student_log_probability)


## What this test proved

After running Part 2 and passing its assertions:

- The student generated the trajectory in Part 1.
- The complete sequence was passed through the same student again.
- At each position, the student predicted the next token.
- We extracted the log probability assigned to the token that actually appeared.
- Prompt-token scores were excluded; the prompt remains the conditioning context.
- We now have one student log probability for every student-generated response token in `response_student_log_probs`.
- We still have not used the teacher or updated any model weights.
- The next step is to have the teacher score the **exact same response tokens**.
